<div style="display: flex; align-items: center;">
  <img src="../Images/Logos/DecisionIntelligenceWorkshopLogo.png" width="60px" style="margin-right: 10px">
  <span style="font-size: 1.5em; font-weight: bold;">Workshop (AI Extensions) - Decisions with Chat Completions & Responses APIs</span>
</div>

Decision Intelligence applied in this module:  
* Introduces the OpenAI Chat Completions API & Responses API as LLM AI interfaces 
* Chat Completions: Composing decisions with system and instruction prompts. Listing of various decision-making frameworks and with their descriptions 
* Chat Completions: Crafting a multi-turn decision scenario 
* Chat Completions: Inspecting the gathered intelligence (chat history)


OpenAI offers two main APIs for building with its models: **Chat Completions** and the **Responses API**. Chat Completions is the established interface for sending conversational messages and receiving model-generated replies, making it useful for chatbots, assistants, and straightforward text generation workflows. The Responses API is the newer, more flexible interface designed to support modern AI applications, including multimodal inputs, structured outputs, tool use, and stateful interactions.

**Chat Completions commonly provides:**

* Message-based chat interactions using roles like system, user, and assistant
* Stateless requests by default; your application sends prior messages each turn
* Full control over what conversation history and context the model receives
* Broad compatibility with existing OpenAI chat-based implementations

**Responses API commonly provides:**

* A unified interface for text, multimodal inputs, tools, and structured outputs
* Built-in stateful context using store: true and prior response chaining
* Easier multi-turn interactions without always resending the full message history
* Better support for deeper reasoning models (GPT-5.4+) 

In this module, the Microsoft Extensions for AI (MEAI) ability to create a Chat experience will be introduced. This is a much richer experience than just sending simple prompts that are stateless and context is forgotten in subsequent requests.

MEAI has first-class support for chat scenarios, where the user talks back and forth with the LLM, the arguments get populated with the history of the conversation. During each new run of the decision chats, the arguments will be provided to the AI with content. This allows the LLM to know the historical context of the conversation.

----
### Step 1 - Initialize Configuration Builder & Build the AI Orchestration

Execute the next two cells to:
* Use the Configuration Builder to load the API secrets.  
* Use the API configuration to build the ChatCompletions orchestrator. 

In [1]:
// Import the required NuGet configuration packages
#r "nuget: Microsoft.Extensions.Configuration, 10.0.9"
#r "nuget: Microsoft.Extensions.Configuration.Json, 10.0.9"
#r "nuget: System.Text.Json, 10.0.9"

using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using System.IO;
using System;

// Load the configuration settings from the local.settings.json and secrets.settings.json files
// The secrets.settings.json file is used to store sensitive information such as API keys
var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// IMPORTANT: You ONLY NEED either Azure OpenAI or OpenAI connection info, not both.
// Azure OpenAI Connection Info
var azureOpenAIEndpoint = config["AzureOpenAI:Endpoint"];
var azureOpenAIAPIKey = config["AzureOpenAI:APIKey"];
var azureOpenAIModelDeploymentName = config["AzureOpenAI:ModelDeploymentName"];
// OpenAI Connection Info 
var openAIAPIKey = config["OpenAI:APIKey"];
var openAIModelId = config["OpenAI:ModelId"];
// AzureOpenAI or OpenAI selection flag
var useAzureOpenAI = bool.Parse(config["AzureOpenAI:UseAzureOpenAI"] ?? "true");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Extensions.Configuration, 10.0.9 Microsoft.Extensions.Configuration.Json, 10.0.9 System.Text.Json, 10.0.9

In [3]:
// Import the Microdoft Extensions AI NuGet Packages
#r "nuget: Microsoft.Extensions.AI, 10.7.0"
#r "nuget: Microsoft.Extensions.AI.Abstractions, 10.7.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.7.0"
// Import Azure & OpenAI NuGet Packages
#r "nuget: Azure.Identity, 1.21.0"
#r "nuget: OpenAI, 2.11.0"

using Azure;
using Microsoft.Extensions.AI;
using Microsoft.Extensions.Configuration;
using OpenAI;
using System.ClientModel;
using System.ComponentModel;
using System.Text.Json;


// Create the IChatClient based on the selected service
IChatClient chatClient;

// Create a new MEAI ChatClient instance
if (useAzureOpenAI)
{
    Console.WriteLine("Using Azure OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey!);

    var azureOpenAIClient = new OpenAIClient(
        apiKeyCredential,
        new OpenAIClientOptions
        {
            Endpoint = new Uri($"{azureOpenAIEndpoint!.TrimEnd('/')}/openai/v1")
        });

    chatClient = azureOpenAIClient.GetChatClient(azureOpenAIModelDeploymentName).AsIChatClient();
}
else
{
    Console.WriteLine("Using OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey);
    var openAIClient = new OpenAIClient(apiKeyCredential);

    // #pragma warning disable OPENAI001
    chatClient = openAIClient.GetChatClient(openAIModelId).AsIChatClient();
}

Installed Packages Azure.Identity, 1.21.0 Microsoft.Extensions.AI, 10.7.0 Microsoft.Extensions.AI.Abstractions, 10.7.0 Microsoft.Extensions.AI.OpenAI, 10.7.0 OpenAI, 2.11.0

Using Azure OpenAI Service


### Step 2 - Chat Completions: Decisions with Prompt Execution Settings 

<img style="display: block; margin: auto;" width ="600px" src="../Images/DecisionIntelligence-Framework-DemingQuote.png">


Using the Microsoft Extensions AI ChatCompletion service is very similar to inovoking a prompt for basic LLM interactions. The chat completion service will provide very similar results to invoking the prompt directly.

In [4]:
// Simple prompt to list some decision frameworks this GenAI LLM is familiar with 
// LLMs are trained on a diverse range of data and can provide insights on a wide range of topics like decision frameworks
// SLMS (smaller LLMs) are trained on a more specific range of data and may not provide insights on all topics
var simpleDecisionPrompt = """
Provide a list of 5 decision frameworks that can help improve the quality of decisions.

Output Format Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower. 
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.
""";

// Execute the prompt against the AI model
var simpleDecisionPromptResponse = await chatClient.GetResponseAsync(simpleDecisionPrompt);
var simpleDecisionPromptResponseText = simpleDecisionPromptResponse.Text;

// Display the response string as Markdown
simpleDecisionPromptResponseText.DisplayAs("text/markdown");

### 1. Decision Matrix

Compare options against consistent criteria such as cost, risk, impact, and feasibility.

- Define the criteria.
- Assign each criterion a weight.
- Score each option.
- Calculate weighted totals.
- Review whether the result matches your judgment.

Useful for: choosing vendors, projects, products, or strategic priorities.


### 2. Cost–Benefit Analysis

Evaluate whether the expected benefits of an option justify its costs.

- Identify direct and indirect costs.
- Estimate tangible and intangible benefits.
- Account for uncertainty and timing.
- Compare alternatives using net value or return on investment.

Useful for: investments, process changes, and resource allocation.


### 3. Premortem Analysis

Assume the decision has failed and work backward to identify why.

- Imagine a future failure.
- List the causes that might have led to it.
- Identify which risks are preventable.
- Add safeguards or adjust the decision.

Useful for: high-stakes decisions, major projects, and risk management.


### 4. OODA Loop

Make decisions through a repeated cycle of observing, interpreting, acting, and learning.

- **Observe:** Gather relevant information.
- **Orient:** Analyze the context and implications.
- **Decide:** Select a course of action.
- **Act:** Implement it and observe the results.

Useful for: fast-changing environments where decisions need continuous adjustment.


### 5. Six Thinking Hats

Examine a decision from several distinct perspectives to reduce bias and improve completeness.

- **White hat:** Facts and information
- **Red hat:** Emotions and intuition
- **Black hat:** Risks and weaknesses
- **Yellow hat:** Benefits and opportunities
- **Green hat:** Alternatives and creativity
- **Blue hat:** Process and next steps

Useful for: team decisions, complex problems, and situations where competing viewpoints matter.

----
### Step 3 - Chat Completions: Decision Chat with a System Prompt & Chat History 

In the previous examples, simple decision prompts were used. In more sophisticated scenarios, a conversational history and state is required to be maintained. The Chat Completions API definition includes mechanisms to maintain state.  

Microsoft Extensions for AI includes a ChatHistory object that can be used with the ChatCompletionService to provide historical chat context to the LLM. Notice that the ChatHistory object differentiates between the different types of chat messages:
* System Message - System or MetaPrompt. These are usually global instructions that set the "overall rules" for interacting with the LLMs.
* User Message - A message from the user
* Assistant Message - A message from the LLM. This is a message generated from an assistant or an agent. 

Identifying the messages from which role (user) it came from can help the LLM improve its own reasoning and decision responses. This is a more sophisticated approach than passing chat history in a long dynamic string. ChatHistory objects can be serialized and persisted into databases as well. This allows a system architect to load chat history dynamically, branch history conversations, re-run or simulate different conversation paths. For decision scenarios, these tools are super helpful. 

In [5]:
// Set the overall system prompt to behave like a decision intelligence assistant (persona)
var systemPrompt = """
You are a Decision Intelligence assistant. 
Assist the user in exploring options, reasoning through decisions, problem-solving, and applying systems thinking to various scenarios. 
Provide structured, logical, and comprehensive advice.

Output Format Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower. 
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

// Simple instruction prompt to list 5 (five) decision frameworks this GenAI LLM is familiar with
var simpleDecisionPrompt = """
Provide five Decision Frameworks that can help improve the quality of decisions.
""";

// Create a new chat history object with proper system and user message roles
var chatMessages = new List<ChatMessage>();
// Add system and user messages to the chat history
var systemMessage = new ChatMessage(ChatRole.System, systemPrompt);
var userMessage = new ChatMessage(ChatRole.User, simpleDecisionPrompt);
chatMessages.Add(systemMessage);
chatMessages.Add(userMessage);


// Execute the chat messages against the AI model
var chatHistoryResponse = await chatClient.GetResponseAsync(chatMessages);
var chatHistoryResponseText = chatHistoryResponse.Text;

// Display the response string as Markdown
chatHistoryResponseText.DisplayAs("text/markdown");

// Capture the Assistant response and add it to the chat history
// This will persist the full conversation for future interactions
var assistantChatMessage = new ChatMessage(ChatRole.Assistant, chatHistoryResponseText);

| # | Decision framework | Best used for | How to apply | Key benefit | Common caution |
|---:|---|---|---|---|---|
| 1 | **Expected Value Analysis** | Decisions involving uncertainty, probabilities, and measurable outcomes | List possible outcomes, estimate the probability and value of each, multiply probability × value, then add the results to compare options | Makes trade-offs and uncertainty explicit | Can create false precision when probabilities or values are unreliable |
| 2 | **Decision Matrix** | Comparing several options against multiple criteria | Define evaluation criteria, assign importance weights, score each option, multiply scores by weights, and compare totals | Provides a transparent and repeatable comparison | Results depend heavily on the selected criteria, weights, and scoring scale |
| 3 | **Pre-Mortem Analysis** | High-stakes decisions with significant downside risk | Imagine the decision has failed, identify likely causes, and add safeguards or modify the plan to address them | Reduces overconfidence and surfaces overlooked risks | Can overemphasize unlikely negative scenarios if not balanced with opportunity analysis |
| 4 | **Reversible vs. Irreversible Decision Framework** | Determining how much time and analysis a decision deserves | Classify the decision by cost of reversal; act quickly on reversible choices and use deeper analysis, consultation, and safeguards for irreversible ones | Allocates decision effort according to potential consequences | Some decisions appear reversible but create hidden long-term commitments |
| 5 | **OODA Loop: Observe, Orient, Decide, Act** | Fast-moving, uncertain, or competitive environments | Gather current information, interpret it in context, choose an action, act, and use the results to update the next cycle | Encourages learning and adaptation rather than waiting for perfect information | Rapid action without adequate observation or reflection can reinforce bad assumptions |

In the next step, an additional prompt instruction will be added to the chat history. From the 5 decision frameworks provided in the chat history, Generative AI is asked to recommed a decision framework best suited for the militaty intelligence community. Notice that the previous chat history is automatically provided to provide additional intelligence context. 

<img style="display: block; margin: auto;" width ="600px" src="../Images/DecisionIntelligence-DecisionExecution-NapoleonQuote.png">

In [6]:
// Add the assistant message to the chat history
chatMessages.Add(assistantChatMessage);

// Note: No reference is made to what previous frameworks were listed.
// Note: Previous context is maintained by the MEAI ChatHistory object 
var simpleDecisionPromptFollowupQuestionPartTwo = """
Which of the 5 decision frameworks listed above is best suited for the military intelligence community?  
Think carefully step by step about what decision frameworks are needed to answer the query.  
Select only the single best framework. 
""";

// Add User message to the chat history
var userFollowupChatMessage = new ChatMessage(ChatRole.User, simpleDecisionPromptFollowupQuestionPartTwo);
chatMessages.Add(userFollowupChatMessage);

// Execute the chat messages against the AI model
var chatHistoryResponseMilitaryIntelligence = await chatClient.GetResponseAsync(chatMessages);
var chatHistoryResponseMilitaryIntelligenceText = chatHistoryResponseMilitaryIntelligence.Text;

// Display the response string as Markdown
chatHistoryResponseMilitaryIntelligenceText.DisplayAs("text/markdown");

| Selection | Best framework | Why it is best suited to the military intelligence community | How it applies |
|---|---|---|---|
| **1** | **OODA Loop: Observe, Orient, Decide, Act** | Military intelligence operates in dynamic, adversarial environments where information is incomplete, time-sensitive, and continuously changing. The OODA Loop supports rapid adaptation, integrates new intelligence as it arrives, and emphasizes acting before an opponent can adjust. | Collect and validate intelligence (**Observe**), assess it within operational and strategic context (**Orient**), select a course of action or intelligence priority (**Decide**), act and monitor the results (**Act**), then repeat the cycle as conditions evolve. |

-----
### Step 4 - Chat Completions: Decision Conversation with a Multi-Turn Conversation Scenario 

<img style="display: block; margin: auto;" width ="600px" src="../Images/DecisionIntelligence-Quote-WarrenBuffet-Value.png"> 

The list of Chat Messages persists the state. This allows for natural converations between the user and AI to be persisted for future use. It can be saved for future reference, it can be replayed, it can be forked in certain spots to try different scenarios, it can be used as a starting point for other conversations etc. 

In the scenario below, let's build a multi-turn conversation about a decision over a purchase for a fitness tracker. Notice that each turn the appropriate Chat History roles are populated with the appropriate User prompts and assistant (AI) responses. This in effect acts as the Gathering Intelligence step of the Decision Intelligence framework as the conversation information has been persisted and can be re-used as context for the decision scenario.

In [7]:
// Set the overall system prompt to behave like a decision intelligence assistant (persona)
var systemPromptDecisionConversation = """
You are a Decision Intelligence assistant. 
Assist the user in exploring options, reasoning through decisions, problem-solving, and applying systems thinking to various scenarios. 
Provide structured, logical, and comprehensive advice.
""";

// User prompt to start a decision conversation about buying a fitness tracker
var userPromptDecisionConversationTurn1 = """
I am considering buying a fitness tracker but I am not sure which features really matter. 
Could you explain the main considerations? 
""";

var assistantResponseDecisionConversationTurn1 = """
Absolutely! Key considerations for choosing a fitness tracker often include: 
- Fitness Features (e.g., step counting, heart-rate monitoring, workout tracking)
- Health Monitoring (like sleep tracking or advanced ECG capabilities)
- Battery Life
- Comfort and Fit
- Compatibility with your smartphone's operating system
- Price Range
- Knowing which of these you value most will help narrow down your options.
""";

var userPromptDecisionConversationTurn2 = """
Got it. Battery life and sleep tracking are both important to me. 
What's a typical battery life range for trackers, and how good is their sleep tracking?
""";

var assistantResponseDecisionConversationTurn2 = """
Different fitness trackers have varying battery lives, typically anywhere from 2-3 days (for more feature-rich or bright-display devices) 
up to about 10-14 days (for more minimalist trackers). 

Regarding sleep tracking:
- Many modern devices can track different sleep stages (light, deep, REM).
- High-end devices even track breathing rate and oxygen saturation during the night.
- Some also offer insights about sleep quality and tips for improvement.

If battery life is a priority, you may want to consider trackers with a more conservative display and fewer power-hungry features.
""";

var userPromptDecisionConversationTurn3 = """
Are there any popular models you would recommend for that balance of battery life and sleep tracking?
""";

var assistantResponseDecisionConversationTurn3 = """
Popular options that strike a balance between robust sleep tracking and good battery life include:
- Fitbit Inspire 3 or Fitbit Charge 5: Both have solid sleep tracking, multi-day battery, and strong app support.
- Garmin Vivosmart 5: Offers decent fitness and sleep insights, plus better-than-average battery life.
- Xiaomi Mi Band 7: Budget-friendly, surprisingly robust health metrics, and a long-lasting battery.

Each varies in price and features, so consider your budget and the specific metrics you’d like to track.
""";

var userPromptDecisionConversationTurn4 = """
That is great infor. 
I also have a friend who said I should consider an Apple Watch. 
Does that have comparable battery life and sleep tracking?
""";

var assistantResponseDecisionConversationTurn4 = """
The Apple Watch is known for its extensive feature set, including advanced health monitoring, 
but its battery life tends to be shorter—usually around 18-24 hours on a single charge 
(or possibly a bit more in low-power mode). While its sleep tracking features have improved over time, 
you might find you need to charge it daily or almost daily, which could conflict with your goal of 24/7 tracking. 
""";

// Create a new chat history with a multi-turn conversation about buying a fitness tracker
// Notice how the conversation flow uses different roles (user and assistant) to simulate a realistic decision-making dialogue
var chatHistoryMessages = new List<ChatMessage>();

chatHistoryMessages.Add(new ChatMessage(ChatRole.System, systemPromptDecisionConversation));
chatHistoryMessages.Add(new ChatMessage(ChatRole.User, userPromptDecisionConversationTurn1));
chatHistoryMessages.Add(new ChatMessage(ChatRole.Assistant, assistantResponseDecisionConversationTurn1));
chatHistoryMessages.Add(new ChatMessage(ChatRole.User, userPromptDecisionConversationTurn2));
chatHistoryMessages.Add(new ChatMessage(ChatRole.Assistant, assistantResponseDecisionConversationTurn2));
chatHistoryMessages.Add(new ChatMessage(ChatRole.User, userPromptDecisionConversationTurn3));
chatHistoryMessages.Add(new ChatMessage(ChatRole.Assistant, assistantResponseDecisionConversationTurn3));
chatHistoryMessages.Add(new ChatMessage(ChatRole.User, userPromptDecisionConversationTurn4));
chatHistoryMessages.Add(new ChatMessage(ChatRole.Assistant, assistantResponseDecisionConversationTurn4));


Using the above conversation history as the "Gathered Intelligence" betweeen the user and the AI assistant, let's make a final decision and ask for final feedback and a decision evaluation. This "Gathered Intelligence" for the decision can be persisted, re-loaded, simulated multiple times, applied to different AI systems to optimize decisions further.  

In [8]:
var userPromptDecisionConversationFinalDecision = """
Thank you for all of that information, I think I'll go with a Fitbit. 
The Fitbit Inspire 3 seems like a good fit for my budget and needs. 

Any quick final thoughts or insights on my decision? 
""";

// Add User message to the chat history
chatHistoryMessages.Add(new ChatMessage(ChatRole.User, userPromptDecisionConversationFinalDecision));

// Execute the chat messages against the AI model
var fitnessTrackerResponse = await chatClient.GetResponseAsync(chatHistoryMessages);
var fitnessTrackerResponseText = fitnessTrackerResponse.Text;

// Add the response to the chat history (Chat Messages)
chatHistoryMessages.Add(new ChatMessage(ChatRole.Assistant, fitnessTrackerResponseText));

// Display the response string as Markdown
fitnessTrackerResponseText.DisplayAs("text/markdown");

The **Fitbit Inspire 3** sounds well matched to your priorities:

- **Battery:** Rated for up to about **10 days**, though frequent workouts, notifications, and screen use can reduce that.
- **Sleep tracking:** Good for trends such as sleep duration, stages, schedule, and sleep score. Treat the stages and sleep-score details as estimates—not medical measurements.
- **Comfort:** Slim and lightweight, making it suitable for overnight wear.
- **Trade-offs:** It lacks built-in GPS, so outdoor route tracking requires your phone. Its screen and smartwatch features are also fairly basic.
- **Subscription:** Check which insights require **Fitbit Premium**; core tracking works without it, but some advanced reports and coaching may be paywalled.
- **Practical tip:** Wear it snugly but comfortably, and charge it during a routine gap—such as while showering—to preserve continuous sleep data.

Overall, if you value long battery life and straightforward sleep tracking more than advanced smartwatch features or standalone GPS, it is a sensible choice.

----
### Step 5 - Chat Completions: Inspect & Optimize Gathered Intelligence of Chat Completion History 

The ChatMessage list is a transparent construct that can be inspected and written out. Because it is a simple list, the chat messages object can be manipulated to replay chats from middle interactions to simulate different outcomes.

Execute the cell below to write out entire decision conversation. 

> **📝 Note:** In the Decision Intelligence framework, chat history can serve as a form of gathered intelligence. Interactions among users, AI models, processes, and agents provide valuable context for effective decision-making. It’s highly recommended to persist chat history objects (agent threads), especially during decision optimization, to ensure critical information is retained and accessible.  

In [9]:
// Print the number of chat interactions and the chat history (turns)
Console.WriteLine("Number of chat interactions: " + chatHistoryMessages.Count());

// Change this to a string builder and show as markdown
var stringBuilderChatHistory = new StringBuilder();
foreach (var message in chatHistoryMessages)
{
    // add a new line for each message
    stringBuilderChatHistory.AppendLine($"**{message.Role.ToString().ToUpper()}**:");
    stringBuilderChatHistory.Append($"{message.Text.Replace("#", string.Empty)}");
    stringBuilderChatHistory.AppendLine("\n");
}

// Display the chat history as Markdown
stringBuilderChatHistory.ToString().DisplayAs("text/markdown");

Number of chat interactions: 11


**SYSTEM**:
You are a Decision Intelligence assistant. 
Assist the user in exploring options, reasoning through decisions, problem-solving, and applying systems thinking to various scenarios. 
Provide structured, logical, and comprehensive advice.

**USER**:
I am considering buying a fitness tracker but I am not sure which features really matter. 
Could you explain the main considerations? 

**ASSISTANT**:
Absolutely! Key considerations for choosing a fitness tracker often include: 
- Fitness Features (e.g., step counting, heart-rate monitoring, workout tracking)
- Health Monitoring (like sleep tracking or advanced ECG capabilities)
- Battery Life
- Comfort and Fit
- Compatibility with your smartphone's operating system
- Price Range
- Knowing which of these you value most will help narrow down your options.

**USER**:
Got it. Battery life and sleep tracking are both important to me. 
What's a typical battery life range for trackers, and how good is their sleep tracking?

**ASSISTANT**:
Different fitness trackers have varying battery lives, typically anywhere from 2-3 days (for more feature-rich or bright-display devices) 
up to about 10-14 days (for more minimalist trackers). 

Regarding sleep tracking:
- Many modern devices can track different sleep stages (light, deep, REM).
- High-end devices even track breathing rate and oxygen saturation during the night.
- Some also offer insights about sleep quality and tips for improvement.

If battery life is a priority, you may want to consider trackers with a more conservative display and fewer power-hungry features.

**USER**:
Are there any popular models you would recommend for that balance of battery life and sleep tracking?

**ASSISTANT**:
Popular options that strike a balance between robust sleep tracking and good battery life include:
- Fitbit Inspire 3 or Fitbit Charge 5: Both have solid sleep tracking, multi-day battery, and strong app support.
- Garmin Vivosmart 5: Offers decent fitness and sleep insights, plus better-than-average battery life.
- Xiaomi Mi Band 7: Budget-friendly, surprisingly robust health metrics, and a long-lasting battery.

Each varies in price and features, so consider your budget and the specific metrics you’d like to track.

**USER**:
That is great infor. 
I also have a friend who said I should consider an Apple Watch. 
Does that have comparable battery life and sleep tracking?

**ASSISTANT**:
The Apple Watch is known for its extensive feature set, including advanced health monitoring, 
but its battery life tends to be shorter—usually around 18-24 hours on a single charge 
(or possibly a bit more in low-power mode). While its sleep tracking features have improved over time, 
you might find you need to charge it daily or almost daily, which could conflict with your goal of 24/7 tracking. 

**USER**:
Thank you for all of that information, I think I'll go with a Fitbit. 
The Fitbit Inspire 3 seems like a good fit for my budget and needs. 

Any quick final thoughts or insights on my decision? 

**ASSISTANT**:
The **Fitbit Inspire 3** sounds well matched to your priorities:

- **Battery:** Rated for up to about **10 days**, though frequent workouts, notifications, and screen use can reduce that.
- **Sleep tracking:** Good for trends such as sleep duration, stages, schedule, and sleep score. Treat the stages and sleep-score details as estimates—not medical measurements.
- **Comfort:** Slim and lightweight, making it suitable for overnight wear.
- **Trade-offs:** It lacks built-in GPS, so outdoor route tracking requires your phone. Its screen and smartwatch features are also fairly basic.
- **Subscription:** Check which insights require **Fitbit Premium**; core tracking works without it, but some advanced reports and coaching may be paywalled.
- **Practical tip:** Wear it snugly but comfortably, and charge it during a routine gap—such as while showering—to preserve continuous sleep data.

Overall, if you value long battery life and straightforward sleep tracking more than advanced smartwatch features or standalone GPS, it is a sensible choice.



By persisting the Decision Intelligence chats and making them available for inspection, can further help the decision-making process. For example you can load the Chat History summarize it or even have AI recommend optimizations for it based on the conversation. 

> **📝 Note:** The process below is simplified to illustrate the power of Gathering Intelligence with user <--> AI interactions. More advanced best practices will be introduced in further sections. 

In [10]:
var userPromptSummarizeTheDecisionChat = """
Briefly summarize the chat conversation about buying a fitness tracker.
""";

chatHistoryMessages.Add(new ChatMessage(ChatRole.User, userPromptSummarizeTheDecisionChat));

// Execute the chat messages against the AI model
var fitnessTrackerDecisionSummaryResponse = await chatClient.GetResponseAsync(chatHistoryMessages);
var fitnessTrackerDecisionSummaryResponseText = fitnessTrackerDecisionSummaryResponse.Text;

// Display the response string as Markdown
fitnessTrackerDecisionSummaryResponseText.DisplayAs("text/markdown");

You prioritized **long battery life and reliable sleep tracking** when choosing a fitness tracker. We discussed typical battery ranges of roughly **2–3 days to 10–14 days**, along with common sleep metrics such as sleep duration, stages, breathing rate, and oxygen saturation.

We compared several options, including Fitbit, Garmin, Xiaomi, and Apple Watch. The Apple Watch offers stronger smartwatch and health features but generally requires daily charging. You decided that the **Fitbit Inspire 3** best fits your budget and needs, while noting its approximately **10-day battery life**, useful sleep trends, lack of built-in GPS, and some Fitbit Premium limitations.

The chat history messsage can further be manipulated by looking at the user to AI interactions and getting feedback for future decision interactions. 

In [12]:
// Optimize the chat history for any future decision interactions
// Prompt to look at the chat history and provide recommendations for optimizing decision-making
var userPromptOptimizeTheDecisionChat = """
Based on the chat conversation so far...
Provide some recommendations on how the decision interactions could be optimized for decision-making.
What are some considerations for the AI to help the use make even better decisions?
""";

// Remove the last message to avoid redundancy
chatHistoryMessages.RemoveAt(chatHistoryMessages.Count - 1);

chatHistoryMessages.Add(new ChatMessage(ChatRole.User, userPromptOptimizeTheDecisionChat));

// Execute the chat messages against the AI model
var decisionOptimizationResponse = await chatClient.GetResponseAsync(chatHistoryMessages);
var decisionOptimizationResponseText = decisionOptimizationResponse.Text;

// Display the response string as Markdown
decisionOptimizationResponseText.DisplayAs("text/markdown");

## How the decision interaction could be improved

The conversation was helpful, but it moved to a recommendation somewhat quickly. A stronger decision process would do the following.

### 1. Clarify the user’s priorities before recommending models

“Battery life” and “sleep tracking” are useful starting points, but the AI should ask:

- What phone do you use—iPhone or Android?
- What is your budget, including any subscription costs?
- Do you need built-in GPS, or is phone-connected GPS enough?
- Do you want notifications, calls, payments, music, or apps?
- Is sleep tracking mainly about sleep duration, sleep stages, snoring/breathing, or identifying possible health concerns?
- Are you comfortable charging every few days, or do you want a week-plus?
- Do you have preferences about size, comfort, appearance, or wristband materials?
- Are there medical conditions or symptoms involved? If so, the device should be framed as a wellness aid, not a diagnostic tool.

These questions help distinguish a simple tracker from a smartwatch and avoid recommending a device that fits one priority while failing another.

### 2. Define measurable decision criteria

The AI could translate preferences into criteria such as:

| Criterion | Example importance |
|---|---:|
| Battery life | Very high |
| Sleep tracking | Very high |
| Comfort overnight | High |
| Cost | High |
| GPS | Low or medium |
| Notifications/apps | Low |
| Subscription requirements | High |
| Phone compatibility | Essential |

Then it could use a simple weighted comparison rather than a general statement that several devices are “good.” This makes the recommendation transparent and easier to revise.

### 3. Distinguish advertised capability from real-world performance

Battery claims such as “up to 10 days” are laboratory or light-use estimates. The AI should explain that actual life depends on:

- Always-on display settings
- Frequent workouts and GPS use
- Notifications
- SpO₂ or other overnight sensors
- Battery age and software updates

Similarly, sleep stages and sleep scores are estimates. Consumer trackers are generally more useful for **trends**—sleep timing, duration, and consistency—than for precise diagnosis of individual sleep stages.

### 4. Compare the relevant alternatives directly

Instead of listing several popular products, the AI could provide a focused comparison:

- **Fitbit Inspire 3:** Strong fit for low cost, light weight, long battery life, and basic sleep trends; lacks built-in GPS and has relatively limited smartwatch features.
- **Fitbit Charge line:** More features and often better exercise tracking, but usually higher cost and potentially shorter battery life.
- **Garmin fitness bands or watches:** Often strong battery life and exercise features; sleep insights and app experience may appeal differently than Fitbit’s.
- **Apple Watch:** Better smartwatch and health ecosystem, but substantially shorter battery life and less convenient overnight charging.
- **Budget bands:** Long battery life and low cost, but app quality, data interpretation, privacy, and long-term support may be weaker.

The comparison should also note model age and availability. Product lines change, and some previously popular models may be discontinued or replaced.

### 5. Include total cost and ecosystem considerations

For Fitbit specifically, the AI should prompt the user to check:

- Whether the device works with their phone and required Fitbit/Google account
- Which features require Fitbit Premium
- Replacement band and charger costs
- Warranty and return policy
- Data export and privacy settings
- Whether the manufacturer is likely to provide app and firmware support

A low purchase price can be less attractive if important insights require an ongoing subscription.

### 6. Address the charging–sleep trade-off explicitly

The key question is not only “How long does the battery last?” but also:

> Can the user charge it regularly without missing the sleep data they care about?

Useful advice would include charging during showering or another daily routine, checking whether the device charges quickly, and considering whether short gaps in data are acceptable.

### 7. Communicate uncertainty and avoid overclaiming

The AI should avoid implying that a tracker can reliably detect medical conditions or precisely measure sleep stages. It should state:

- Sleep tracking is useful for personal trends, not diagnosis.
- Oxygen saturation and breathing metrics may be estimates and can be affected by fit and movement.
- If the user has persistent insomnia, loud snoring, or suspected sleep apnea, a clinician or formal sleep evaluation is more appropriate.

## Example of a better decision interaction

A stronger sequence would be:

1. **Identify goals:** “You prioritize overnight wear, long battery life, and useful sleep trends.”
2. **Ask constraints:** Phone type, budget, GPS needs, subscription tolerance, and desired smartwatch functions.
3. **Rank criteria:** Battery and sleep tracking are essential; GPS and apps may be secondary.
4. **Shortlist two or three devices:** Include only models compatible with those constraints.
5. **Show trade-offs:** Battery, real-world sleep features, subscription costs, comfort, and limitations.
6. **Test the recommendation:** Encourage checking current pricing, app requirements, return policy, and user reviews focused on sleep comfort and battery performance.
7. **Give a conditional recommendation:** For example, “Choose the Inspire 3 if you want a lightweight, low-cost tracker and do not need standalone GPS; choose an Apple Watch only if smartwatch functions outweigh daily charging.”

Overall, the Fitbit Inspire 3 recommendation was reasonable based on the stated priorities, but the decision would be more robust if it were based on explicitly ranked criteria, current product information, total ownership cost, and the user’s phone and feature requirements.